In [ ]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [ ]:
df = pd.read_csv("/content/sarcasm_mixed_reviews_1000.csv")

df = df[["reviews", "style"]].dropna()
df = df.rename(columns={"reviews": "text", "style": "label"})

print(df.head())
print(df["label"].value_counts())

                                                text      label
0  What an annoying outcome: the tablet actually ...  sarcastic
1  So many products on the market. This by far is...   standard
2  Really easy to work with lots of extras. Good ...   standard
3  A stunning display of modern engineering — thi...  sarcastic
4  Ever since I had my fire TV jail broken and pr...   standard
label
sarcastic    500
standard     500
Name: count, dtype: int64


In [ ]:
label2id = {"standard": 0, "sarcastic": 1}
id2label = {0: "standard", 1: "sarcastic"}

df["label"] = df["label"].map(label2id)

In [ ]:
df.head()

,text,label
0,What an annoying outcome: the tablet actually ...,1
1,So many products on the market. This by far is...,0
2,Really easy to work with lots of extras. Good ...,0
3,A stunning display of modern engineering — thi...,1
4,Ever since I had my fire TV jail broken and pr...,0


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 800
Test size: 200


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
test_dataset

Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200
})

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_sarcasm",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch", # Added to match eval_strategy
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.202586,0.029701,0.995000,0.995000
2,0.001288,0.034340,0.995000,0.995000
3,0.000626,0.035686,0.995000,0.995000
4,0.000481,0.036258,0.995000,0.995000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=400, training_loss=0.05124521596357226, metrics={'train_runtime': 135.6172, 'train_samples_per_second': 23.596, 'train_steps_per_second': 2.949, 'total_flos': 210488844288000.0, 'train_loss': 0.05124521596357226, 'epoch': 4.0})

In [ ]:
results=trainer.evaluate()
print(results)

{'eval_loss': 0.029730916023254395, 'eval_accuracy': 0.995, 'eval_f1': 0.9949998749968749, 'eval_runtime': 1.5495, 'eval_samples_per_second': 129.072, 'eval_steps_per_second': 16.134, 'epoch': 4.0}


In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=["standard", "sarcastic"]))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))


Classification Report:

              precision    recall  f1-score   support

    standard       1.00      0.99      0.99       100
   sarcastic       0.99      1.00      1.00       100

    accuracy                           0.99       200
   macro avg       1.00      0.99      0.99       200
weighted avg       1.00      0.99      0.99       200


Confusion Matrix:

[[ 99   1]
 [  0 100]]


In [ ]:
import torch

def predict_sarcasm(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return id2label[pred]

# YOUR TEST CASES
sample_reviews = [
    "This product is fantastic and works perfectly.",
    "Wow, amazing charger. It stopped working in one day.",
    "Absolutely brilliant. I love how it stopped working immediately.",
    "The product is average, not too bad and not too good.",
    "disappointing charger. I thought it would stop working in one day, but is working absolutely fine."
]

for review in sample_reviews:
    print("Review:", review)
    print("Predicted:", predict_sarcasm(review))
    print("-"*60)


Review: This product is fantastic and works perfectly.
Predicted: standard
------------------------------------------------------------
Review: Wow, amazing charger. It stopped working in one day.
Predicted: standard
------------------------------------------------------------
Review: Absolutely brilliant. I love how it stopped working immediately.
Predicted: sarcastic
------------------------------------------------------------
Review: The product is average, not too bad and not too good.
Predicted: standard
------------------------------------------------------------
Review: disappointing charger. I thought it would stop working in one day, but is working absolutely fine.
Predicted: standard
------------------------------------------------------------


Testing zero shot bert for sarcasm

In [ ]:
!pip install transformers -q

from transformers import pipeline
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
df = pd.read_csv("/content/sarcasm_mixed_reviews_1000.csv")

df = df[["reviews", "style"]].dropna()

print(df.head())

                                             reviews      style
0  What an annoying outcome: the tablet actually ...  sarcastic
1  So many products on the market. This by far is...   standard
2  Really easy to work with lots of extras. Good ...   standard
3  A stunning display of modern engineering — thi...  sarcastic
4  Ever since I had my fire TV jail broken and pr...   standard


In [ ]:
labels = ["sarcastic", "standard"]

In [ ]:
sample_reviews = [
    "This product is fantastic and works perfectly.",
    "Wow, amazing charger. It stopped working in one day.",
    "Absolutely brilliant. I love how it stopped working immediately.",
    "The product is average, not too bad and not too good.",
    "disappointing charger. I thought it would stop working in one day, but is working absolutely fine."
]

for r in sample_reviews:
    result = classifier(r, labels)
    print("Review:", r)
    print("Prediction:", result["labels"][0])
    print("Scores:", dict(zip(result["labels"], result["scores"])))
    print("-"*60)

Review: This product is fantastic and works perfectly.
Prediction: standard
Scores: {'standard': 0.9846423864364624, 'sarcastic': 0.015357679687440395}
------------------------------------------------------------
Review: Wow, amazing charger. It stopped working in one day.
Prediction: sarcastic
Scores: {'sarcastic': 0.852803111076355, 'standard': 0.14719687402248383}
------------------------------------------------------------
Review: Absolutely brilliant. I love how it stopped working immediately.
Prediction: sarcastic
Scores: {'sarcastic': 0.8153532147407532, 'standard': 0.18464677035808563}
------------------------------------------------------------
Review: The product is average, not too bad and not too good.
Prediction: standard
Scores: {'standard': 0.9915837645530701, 'sarcastic': 0.00841626524925232}
------------------------------------------------------------
Review: disappointing charger. I thought it would stop working in one day, but is working absolutely fine.
Prediction: 

In [ ]:
predictions = []

for text in df["reviews"]:
    result = classifier(text, labels)
    predictions.append(result["labels"][0])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
print("Accuracy:", accuracy_score(df["style"], predictions))

print("\nClassification Report:\n")
print(classification_report(df["style"], predictions))

Accuracy: 0.613

Classification Report:

              precision    recall  f1-score   support

   sarcastic       0.64      0.52      0.57       500
    standard       0.59      0.71      0.65       500

    accuracy                           0.61      1000
   macro avg       0.62      0.61      0.61      1000
weighted avg       0.62      0.61      0.61      1000

